In order to help the readers, we want to analyze the data of the <i>Goodreads</i> website and extract interesting and useful information from it.

The information of books is in the <code>books_db.csv</code> file, the authors in the <code>authors.csv</code> file, the publishers in the <code>publisher.csv</code> file, and the languages in the <code>language.csv</code> file.

In [4]:
#Let's read the data
import pandas as pd
import numpy as np

df = pd.read_csv('books_db.csv')
authors =pd.read_csv('authors.csv')
language =pd.read_csv('language.csv')
publisher =pd.read_csv('publisher.csv')

As you can see in the main dataset (books), the values of the three columns <code>lang_id</code>, <code>author_id</code>, and <code>publisher_id</code> are ids only.<code></code> These IDs correspond to the number of profiles in other dataframes. In this section, we intend to extract their complete information from the relevant dataframe and add it to the <code>df</code> dataframe.

In [5]:
#Let's combine them by merging.
df=pd.merge(df,authors,left_on='author_id',right_index=True)
df=pd.merge(df,language,left_on='lang_id',right_index=True)
df=pd.merge(df,publisher,left_on='publisher_id',right_index=True)

#Let's drop the columns that we don't need.
df.drop(columns = ['author_id', 'lang_id', 'publisher_id'], inplace = True)

In [6]:
#Just a little bit of sorting columns.
df.columns=['bookID','title','average_rating','isbn','isbn13','num_pages','ratings_count','text_reviews_count','publication_date','author','lang','publisher']
df.head(3)

,bookID,title,average_rating,isbn,isbn13,num_pages,ratings_count,text_reviews_count,publication_date,author,lang,publisher
0,1,Harry Potter and the Half-Blood Prince (Harry ...,4.57,439785960,9.780440e+12,652,2095690,27591,9/16/2006,J.K. Rowling,eng,Scholastic Inc.
1,2,Harry Potter and the Order of the Phoenix (Har...,4.49,439358078,9.780440e+12,870,2153167,29221,9/1/2004,J.K. Rowling,eng,Scholastic Inc.
2,4,Harry Potter and the Chamber of Secrets (Harry...,4.42,439554896,9.780440e+12,352,6333,244,11/1/2003,J.K. Rowling,eng,Scholastic


Now we calculate the average score of the books of each publication and save the result in the <code>publisher_rating</code> variable.

In [7]:
publisher_rating =df.groupby('publisher')['average_rating'].agg('mean')
publisher_rating

publisher
18-Oct                           3.650000
1st Book Library                 3.580000
1st World Library                4.180000
A & C Black (Childrens books)    4.400000
A Harvest Book/Harcourt  Inc.    4.070000
                                   ...   
聯經出版事業股份有限公司                     4.400000
英文漢聲出版股份有限公司                     4.220000
角川書店 (Kadokawa Shoten)           3.815000
講談社                              4.283333
集英社                              4.237500
Name: average_rating, Length: 2290, dtype: float64

In this section let's add a column called <code>num_pages_grp</code> to the dataframe, in which it shows the number of pages of the book in the following intervals:

In [8]:
condition = [
	(df['num_pages'] < 250),
	(df['num_pages'] >= 250) & (df['num_pages'] <= 499),
	(df['num_pages'] >= 500) & (df['num_pages'] <= 749),
	(df['num_pages'] >= 750) & (df['num_pages'] <= 999),
	(df['num_pages'] >= 1000)
]
choices = ['<250', '250-499', '500-749', '750-999', '>=1000']
df['num_pages_grp'] = np.select(condition, choices,default='out of range')
df.head()

,bookID,title,average_rating,isbn,isbn13,num_pages,ratings_count,text_reviews_count,publication_date,author,lang,publisher,num_pages_grp
0,1,Harry Potter and the Half-Blood Prince (Harry ...,4.57,439785960,9.780440e+12,652,2095690,27591,9/16/2006,J.K. Rowling,eng,Scholastic Inc.,500-749
1,2,Harry Potter and the Order of the Phoenix (Har...,4.49,439358078,9.780440e+12,870,2153167,29221,9/1/2004,J.K. Rowling,eng,Scholastic Inc.,750-999
2,4,Harry Potter and the Chamber of Secrets (Harry...,4.42,439554896,9.780440e+12,352,6333,244,11/1/2003,J.K. Rowling,eng,Scholastic,250-499
3,5,Harry Potter and the Prisoner of Azkaban (Harr...,4.56,043965548X,9.780440e+12,435,2339585,36325,5/1/2004,J.K. Rowling,eng,Scholastic Inc.,250-499
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,4.78,439682584,9.780440e+12,2690,41428,164,9/13/2004,J.K. Rowling,eng,Scholastic,>=1000


Now we want to know how many books each author has written in each of these periods.

In [9]:
author_page=df.groupby(['author' , 'num_pages_grp'])['title'].agg('count')
author_page

author                 num_pages_grp
A.B. Yehoshua          250-499          1
                       500-749          1
A.D.P. Briggs          250-499          1
A.E. Cunningham        <250             1
A.J. Jacobs            250-499          1
                                       ..
bell hooks             <250             5
Åsne Seierstad         250-499          3
Émile Zola             250-499          6
                       500-749          3
Éric-Emmanuel Schmitt  <250             1
Name: title, Length: 5547, dtype: int64